In [4]:
import numpy as np
import sys
import math
import time
import random
import json

In [9]:
#FUNCIONES DE GENERACIÓN DE INSTANCIAS
def sat_instance_generator(n, m): #generar instancias de QSAT2 con n cláusulas y m variables en cada cláusula 
    sat_instance_descriptor=[]
    unused_x=list(range(n))
    unused_y=list(range(n))
    
    for i in range(n):
        
        clause_i=[]
        for j in range(m):
            
            if unused_x and unused_y:   #asegurar que se usan todas las variables
                a=random.randint(0,1)
                if not a:
                    c=[a,random.choice(unused_x)]
                else:
                    c=[a,random.choice(unused_y)]
            elif unused_x and not unused_y:
                c=[0,random.choice(unused_x)]
            elif not unused_x and unused_y:
                c=[1,random.choice(unused_y)]
            else: 
                c=[random.randint(0,1),random.randint(0,n-1)]
                
            if j==1: 
                while c==clause_i[0]:   #asegurar que no se repitan variables en la misma cláusula
                    c=[random.randint(0,1),random.randint(0,n-1)]
            elif j==2:
                if clause_i[0][0]==clause_i[1][0]==c[0]==0:  #asegurar que no haya cláusulas solo con x, pero sí solo con y
                    c=[int(not c[0]), c[1]]
                while c==clause_i[0] or c==clause_i[1]:   #asegurar que no se repitan variables en la misma cláusula
                    c=[random.randint(0,1),random.randint(0,n-1)]
    
            clause_i.append(c)
    
            if c[0]==0:   #llevar la cuenta de que variables quedan sin usar
                if c[1] in unused_x:
                    del unused_x[unused_x.index(c[1])]
                    
            if c[0]==1:
                if c[1] in unused_y:
                    del unused_y[unused_y.index(c[1])]

        negations=[]
        for i in range(m):
            negations.append(random.randint(0,1))
        clause_i.append(negations)
        
        sat_instance_descriptor.append(clause_i)
        
    return sat_instance_descriptor

def sat_instance_instancer(x, y, sat_instance_descriptor):    #para una instancia de QSAT2 y unos valores de x e y, instancia el problema
    sat_instanced=[]
        
    for clause_i in sat_instance_descriptor:
        clause_i_instanced=[] 
        
        for i in range(len(clause_i)-1):
            if clause_i[i][0]==0:
                if clause_i[-1][i]:
                    clause_i_instanced.append(x[clause_i[i][1]])
                else:
                    clause_i_instanced.append(int(not(x[clause_i[i][1]])))
            else:
                if clause_i[-1][i]:
                    clause_i_instanced.append(y[clause_i[i][1]])
                else:
                    clause_i_instanced.append(int(not(y[clause_i[i][1]])))

        sat_instanced.append(clause_i_instanced)
        
    return sat_instanced

def random_x_y(n, t=1):   #genera individuos aleatorios de tamaño n
    xs=[]
    ys=[]
    for i in range(t):
        x=[]
        y=[]
        for i in range(n):
            x.append(random.randint(0,1))
            y.append(random.randint(0,1))
        xs.append(x)
        ys.append(y)
    return xs, ys

def sat_instance_generator_clause_variabe(n, m, nc, fair=True): #instancias de QSAT2 con n cláusulas y m variables en cada cláusula 
    sat_instance_descriptor=[]
    unused_x=list(range(n))
    unused_y=list(range(n))
    
    for i in range(nc):
        clause_i=[]
        for j in range(m):
            
            if unused_x and unused_y:   #asegurar que se usan todas las variables
                a=random.randint(0,1)
                if not a:
                    c=[a,random.choice(unused_x)]
                else:
                    c=[a,random.choice(unused_y)]
            elif unused_x and not unused_y:
                c=[0,random.choice(unused_x)]
            elif not unused_x and unused_y:
                c=[1,random.choice(unused_y)]
            else: 
                c=[random.randint(0,1),random.randint(0,n-1)]
                
            if j==1: 
                while c==clause_i[0]:   #asegurar que no se repitan variables en la misma cláusula
                    c=[random.randint(0,1),random.randint(0,n-1)]
            elif j==2:
                if clause_i[0][0]==clause_i[1][0]==c[0]==0:  #asegurar que no haya cláusulas solo con x, pero sí solo con y
                    c=[int(not c[0]), c[1]]
                while c==clause_i[0] or c==clause_i[1]:   #asegurar que no se repitan variables en la misma cláusula
                    c=[random.randint(0,1),random.randint(0,n-1)]
    
            clause_i.append(c)
    
            if c[0]==0:   #llevar la cuenta de que variables quedan sin usar
                if c[1] in unused_x:
                    del unused_x[unused_x.index(c[1])]
                    
            if c[0]==1:
                if c[1] in unused_y:
                    del unused_y[unused_y.index(c[1])]

        negations=[]
        for i in range(m):
            negations.append(random.randint(0,1))
        clause_i.append(negations)
        
        sat_instance_descriptor.append(clause_i)
        
    return sat_instance_descriptor

In [3]:
def fitness_check_FNC(sat_instanced):    #comprueba el número de cláusulas no aceptadas de un problema instanciado (fórmula SAT en FNC)
    fitness=len(sat_instanced)
    for clause_i in sat_instanced:
        if 1 in clause_i:
            fitness-=1
    return fitness
    
def fitness_check_FND(sat_instanced):    #comprueba el número de cláusulas no aceptadas de un problema instanciado (fórmula SAT en FND)
    fitness=0
    for clause_i in sat_instanced:
        if 0 in clause_i:
            fitness+=1
    return fitness

In [5]:
#Funciones para guardar y cargar instancias y datos en archivos JSON
def save_instance_file(sat_instance_descriptor, file_path="QSAT2_instance.json"):
    json_str = json.dumps(sat_instance_descriptor)
    file_path 
    with open(file_path, 'w') as file:
        json.dump(json_str, file)
    return 0

def load_file_instance(file_path='QSAT2_instance.json'):
    file = open(file_path,)
    data = json.load(file)
    array = json.loads(data)
    return array

In [11]:
#FUNCIONES DE RESOLUBILIDAD DE INSTANCIAS
#Función recursiva que genera todos las cadenas binarias de tamaño len(s)
def binstrRec(s, i, res):
    n = len(s)

    if i == n:
        res.append("".join(s))
        return
        
    s[i] = '0'
    binstrRec(s, i + 1, res)

    s[i] = '1'
    binstrRec(s, i + 1, res)
    
#Función que devuelve todas las cadenas binarias de longitud n
def binstr(n):
    s = ['0'] * n
    res = []
    binstrRec(s, 0, res)
    return res

def full_tree_exploration_opt(xs, ys, n=10, m=3, file=None):   #genera una instancia de QSAT2 y explora todas las combinaciones
    result=[]
    if not file:
        sat_instance_descriptor=sat_instance_generator(n,m)
    else:
        sat_instance_descriptor=load_file_instance(file)
    for x in xs:
        for y in ys:
            sat_instanced=sat_instance_instancer(x, y, sat_instance_descriptor)
            fitness=fitness_check_FND(sat_instanced)
            result.append([x,y,fitness])
    return sat_instance_descriptor, result

def check_succesful_tree_branches_FND_opt(xs, ys, sat_size, result):   #comprueba si existe una rama x para la que todas sus ramas y satifcacen la instancia de SAT
    valid_x=[]
    res_index=0
    for i in range(len(xs)):
        valid_branch=1
        for j in range(len(ys)):
            if result[res_index][2]==sat_size:
                valid_branch=0 
            res_index+=1   
        if valid_branch:
            valid_x.append(result[res_index-1][0])
    return valid_x

def brute_force_QSAT2_FND_opt(n=10, m=3, file=None):   #genera todos los posibles individuos de x e y, genera un problema QSAT2, lo resuleve para todas las combinaciones y comprueba la satisfactibilidad para las subramas del árbol 
    xs=[]
    x=[]
    a=binstr(n)
    for string in a:
        for i in range(len(string)):
            x.append(int(string[i]))
        xs.append(x)
        x=[]
    ys=xs.copy()
    tree=full_tree_exploration_opt(xs, ys, n, m, file)
    valid_branches=check_succesful_tree_branches_FND_opt(xs, ys, len(tree[0]), tree[1])
    return tree[0], tree[1],  valid_branches

def full_tree_exploration_simple(xs, ys, sat_instance_descriptor, n=10, m=4):   #genera una instancia de QSAT2 y explora todas las combinaciones
    result=[]
    for x in xs:
        for y in ys:
            sat_instanced=sat_instance_instancer(x, y, sat_instance_descriptor)
            fitness=fitness_check_FND(sat_instanced)
            result.append([x,y,fitness])
    return sat_instance_descriptor, result

def brute_force_QSAT2_simple(xs, sat_instnace_descriptor, n=10, m=4):   #recibe las ramas de x a comprobar, genera todos los posibles individuos de y, genera un problema QSAT2, lo resuleve para todas las combinaciones y comprueba la satisfactibilidad para las subramas del árbol 
    ys=[]
    y=[]
    a=binstr(n)
    for string in a:
        for i in range(len(string)):
            y.append(int(string[i]))
        ys.append(y)
        y=[]
    tree=full_tree_exploration_simple(xs, ys, sat_instnace_descriptor, n, m)
    valid_branches=check_succesful_tree_branches_FND_opt(xs, ys, len(tree[0]), tree[1])
    return tree[0], tree[1], valid_branches

def full_tree_exploration_net(xs, ys, n=10, m=3, file=None):   #genera una instancia de QSAT2 y explora todas las combinaciones
    result=[]
    if not file:
        sat_instance_descriptor=sat_instance_generator(n,m)
    else:
        sat_instance_descriptor=load_file_instance(file)
    for x in xs:
        for y in ys:
            sat_instanced=sat_instance_instancer(x, y, sat_instance_descriptor)
            fitness=fitness_check_FND(sat_instanced)
            result.append([x,y,fitness])
    return sat_instance_descriptor, result


def brute_force_QSAT2_FND_net(n=10, m=3, nc=10, file=None):   #genera todos los posibles individuos de x e y, genera un problema QSAT2, lo resuleve para todas las combinaciones y comprueba la satisfactibilidad para las subramas del árbol 
    xs=[]
    x=[]
    a=binstr(n)
    for string in a:
        for i in range(len(string)):
            x.append(int(string[i]))
        xs.append(x)
        x=[]
    ys=xs.copy()
    tree=full_tree_exploration_net(xs, ys, n, m, file)
    resultado=0
    for res in tree[1]:
        if(res[2]<nc):
            resultado+=1
    return tree[0], tree[1], resultado

In [13]:
#Función para generación de conjuntos de instancias con un tamaño y ratio concretos de cláusulas-variables, se comprueba su equilibrio
def batch_test_clause_var_ratio_net(n=10, m=3, ratio=1, file_generic_name="QSAT2_FND_instance_n10_c1_net_test", acceptance=0.05, verbose=True, num_instance_gen=100):
    nc=int(2*n*ratio)
    len_max=pow(2,n*2)
    file_generic=file_generic_name+".json"
    resultados=[]
    
    for i in range(num_instance_gen):
        sat_instance_descriptor=sat_instance_generator_clause_variabe(n, m, nc)
        save_instance_file(sat_instance_descriptor, file_generic)
        t1=time.time()
        resultado_full_n10_c_net=brute_force_QSAT2_FND_net(n,m,nc,file_generic)
        t2=time.time()
        t=t2-t1
        resultados.append([i, resultado_full_n10_c_net[2], t])
        if(verbose):
            print(resultados[-1])
        if((resultados[-1][1]<(len_max/2 +len_max*acceptance)) and (resultados[-1][1]>(len_max/2 -len_max*acceptance))):
            file_name=file_generic_name+"_"+str(i)+".json"
            save_instance_file(sat_instance_descriptor, file_name)
            file_name2=file_generic_name+"_"+str(i)+"_full_solution.json"
            save_instance_file(resultado_full_n10_c_net[2], file_name2)
            if(verbose):
                print(file_name)
    
    return resultados

#Función para generación de conjuntos de instancias con un tamaño concreto y ratio de cláusulas-variables en un intervalo, se comprueba su equilibrio
def batch_test_clause_var_ratio_interval_net(n=10, m=3, ratio=1, ratio_max=2, intervalo=0.02, file_generic="QSAT2_FND_instance_n10_claus_var_net_test_1to2.json", acceptance=0.05, verbose=True):
    resultados=[]   
    while(ratio<=(ratio_max+intervalo)):
        nc=int(2*n*ratio)
        len_max=pow(2,n*2)
        resultados.append([ratio, []])
        sumatorio=0
        contador=0
        for i in range(100):
            sat_instance_descriptor=sat_instance_generator_clause_variabe(n, m, nc)
            save_instance_file(sat_instance_descriptor, file_generic)
            t1=time.time()
            resultado_full_n10_c_net=brute_force_QSAT2_FND_net(n,m,nc,file_generic)
            t2=time.time()
            t=t2-t1
            resultados[-1][1].append([i, resultado_full_n10_c_net[2], t])
            sumatorio+=resultados[-1][1][-1][1]
            if((resultados[-1][1][-1][1]< (len_max/2 + len_max*acceptance)) and (resultados[-1][1][-1][1]> (len_max/2 - len_max*acceptance))):
                contador+=1
    
        resultados[-1].append(contador/100)
        resultados[-1].append(sumatorio/100)
        
        ratio+=intervalo
        
        if(verbose):
            print(resultados[-1])
            
    return resultados

#Función para comprobar si un conjunto específico de instancias generadas con los métodos anteriores son resolubles (pasando los índices de los archivos)
def batch_check_brute_force_solvable(file_index, n=10, m=4, file_generic_name="QSAT2_FND_instance_n10_m4_c055_net_test", file_generic_sol_name="QSAT2_FND_instance_n10_m4_c055", verbose=True ):
    resultados=[]
    for i in file_index:
        filename=file_generic_name+"_"+str(i)+".json"
        instance=load_file_instance(filename)
        t1=time.time()
        resultado_full_n10=brute_force_QSAT2_FND_opt(n,m,filename)
        t2=time.time()
        t=t2-t1
        resultados.append([i, filename, len(resultado_full_n10[2]), t])
        if(verbose):
            print(resultados[-1])
        if(len(resultado_full_n10[2])>0):
            solname=file_generic_sol_name+"_"+str(i)+"_full_solution.json"
            save_instance_file(resultado_full_n10[2], solname)
            resultados[-1].append(solname)
            if(verbose):
                print(solname)

    return resultados

#Pipeline final completa para generación de conjuntos de num_instances_gen_objective instancias equilibradas y resoloubles
def full_batch_clause_var_ratio_net_instance_generation(n=10, m=4, ratio=0.55, file_generic_name="QSAT2_FND_instance", acceptance=0.05,  num_instances_gen_objective=60, verbose=True):
    nc=int(2*n*ratio)
    len_max=pow(2,n*2)
    file_generic=file_generic_name+".json"
    resultados_net=[]
    resultados_solvable=[]
    resultados=[]
    i=0
    num_valid_instances=0

    t1_global=time.time()

    while(num_valid_instances<num_instances_gen_objective):
    #for i in range(num_instance_gen):
        sat_instance_descriptor=sat_instance_generator_clause_variabe(n, m, nc)   #se genera la instancia de QSAT2
        save_instance_file(sat_instance_descriptor, file_generic)
        t1=time.time()
        resultado_full_net=brute_force_QSAT2_FND_net(n,m,nc,file_generic)   #se comprueba el numero de soluciones netas de la instancia
        t2=time.time()
        t=t2-t1
        resultados_net.append([i, resultado_full_net[2], t])
        resultados.append(resultados_net[-1])
        if(verbose):
            print(resultados_net[-1])
        if((resultados_net[-1][1]<(len_max/2 +len_max*acceptance)) and (resultados_net[-1][1]>(len_max/2 -len_max*acceptance))):   #si el número de soluciones netas está en el rango
            file_name=file_generic_name+"_"+str(i)+".json"
            save_instance_file(sat_instance_descriptor, file_name)     #se guarda la instancia
            resultados[-1].append(file_name)
            #file_name2=file_generic_name+"_"+str(i)+"_full_solution_net.json"
            #save_instance_file(resultado_full_net[2], file_name2)
            if(verbose):
                print(file_name)

            t1_solv=time.time()
            check_solvable_resultado_full=brute_force_QSAT2_FND_opt(n,m,file_name)    #se calcula el número de soluciones válidas 
            t2_solv=time.time()
            t_solv=t2_solv-t1_solv
            resultados_solvable.append([i, len(check_solvable_resultado_full[2]), t_solv])
            resultados[-1].append([len(check_solvable_resultado_full[2]), t_solv])
            if(verbose):
                print(resultados_solvable[-1])
            if(len(check_solvable_resultado_full[2])>0):    #y se comprueba si es resoluble
                full_sol_name=file_generic_name+"_"+str(i)+"_full_solution.json"
                save_instance_file(check_solvable_resultado_full[2], full_sol_name)   #si tiene alguna solucion valida, se guardan sus soluciones validas
                resultados[-1].append(full_sol_name)
                num_valid_instances+=1
                if(verbose):
                    print(full_sol_name)

        t2_global=time.time()
        t_global=t2_global-t1_global
        if(verbose):
            print(i, " --- ", num_valid_instances, "/", num_instances_gen_objective, " --- ", t_global, '\n')
        i+=1
    
    return resultados


In [15]:
#ALGORITMO GENÉTICO BINIVEL

#FUNCIONES DE CRUCE
def cruce_1P(population, prob_mut=0.01):   #función de cruce en un punto
    indices_pob=[]
    hijos=[]
   
    tamanho=len(population[0])
     
    for i in range(len(population)):
        indices_pob.append(i)
        
    for i in range(int((len(population))/2)):
        
        p1=random.choice(indices_pob)
        padre1= population[p1]
        indices_pob.remove(p1)
        p2=random.choice(indices_pob)
        padre2=population[p2]
        indices_pob.remove(p2)
        
        punto_corte=random.choice(list(range(tamanho-1))[1:])
        padre1_1=padre1[:punto_corte]
        padre1_2=padre1[punto_corte:]
        padre2_1=padre2[:punto_corte]
        padre2_2=padre2[punto_corte:]
        hijo1=padre1_1 + padre2_2
        hijo2=padre2_1 + padre1_2
        
        for i in range(len(hijo1)):
            muta1=random.random()
            muta2=random.random()
            if muta1<=prob_mut:
                hijo1[i]=int(not hijo1[i])
            if muta2<=prob_mut:
                hijo2[i]=int(not hijo2[i])
        hijos.append(hijo1)
        hijos.append(hijo2)
        
        #print(padre1, padre2, punto_corte)
        #print(hijo1, hijo2)
    
    return hijos

def cruce_2P(population, prob_mut=0.01):    #función de cruce en 2 puntos
    indices_pob=[]
    hijos=[]
   
    tamanho=len(population[0])
     
    for i in range(len(population)):
        indices_pob.append(i)
        
    for i in range(int((len(population))/2)):
        
        p1=random.choice(indices_pob)
        padre1= population[p1]
        indices_pob.remove(p1)
        p2=random.choice(indices_pob)
        padre2=population[p2]
        indices_pob.remove(p2)

        posibles_puntos_corte=list(range(tamanho-1))[1:]
        punto_corte1=random.choice(posibles_puntos_corte[:-2])
        for j in range(len(posibles_puntos_corte)):
            if j<punto_corte1: 
                del posibles_puntos_corte[0]
        punto_corte2=random.choice(posibles_puntos_corte[1:])
        
        #print(posibles_puntos_corte)
        
        padre1_1=padre1[:punto_corte1]
        padre1_2=padre1[punto_corte1:punto_corte2]
        padre1_3=padre1[punto_corte2:]
        
        padre2_1=padre2[:punto_corte1]
        padre2_2=padre2[punto_corte1:punto_corte2]
        padre2_3=padre2[punto_corte2:]
        
        hijo1=padre1_1 + padre2_2 + padre1_3
        hijo2=padre2_1 + padre1_2 + padre2_3
        
        for i in range(len(hijo1)):
            muta1=random.random()
            muta2=random.random()
            if muta1<=prob_mut:
                hijo1[i]=int(not hijo1[i])
            if muta2<=prob_mut:
                hijo2[i]=int(not hijo2[i])
        hijos.append(hijo1)
        hijos.append(hijo2)
        
        #print(padre1, padre2, punto_corte1, punto_corte2)
        #print(hijo1, hijo2)
    
    return hijos

#FUNCIONES DE SELECCIÓN PARA Ys
def selection_elitism_ys(x, y_pop, y_hijos, fit_list, sat_instance_descriptor):   #función de selección elitista para las ys, cuanto mayor sea el fitness, más cláusulas rompe el individuo y para las variables xs fijas
    new_gen=[]
    new_fit_list=[]
    tam_pop=len(y_pop)
    
    for y_hijo in y_hijos:    #para cada hijo se calcula su fitness
        fit=fitness_check_FND(sat_instance_instancer(x, y_hijo, sat_instance_descriptor))
        y_pop.append(y_hijo)
        fit_list.append(fit)
        
    for i in range(len(y_pop)):    #se ordenan los individuos por su fitness
        y_pop[i]=([y_pop[i], fit_list[i]])
    y_pop_sorted=list(reversed(sorted(y_pop, key=lambda ind: ind[1])))
    
    for i in range(tam_pop):   #los individuos y con la mayor fitness sobreviven
       new_gen.append(y_pop_sorted[i][0])
       new_fit_list.append(y_pop_sorted[i][1])
        
    return new_gen, new_fit_list

def selection_tourney_ys(x, y_pop, y_hijos, fit_list, sat_instance_descriptor):   #función de selección por torneo
    #selección ys, torneo
    new_gen=[]
    new_fit_list=[]
    tam_pop=len(y_pop)
    
    for y_hijo in y_hijos:
        fit=fitness_check_FND(sat_instance_instancer(x, y_hijo, sat_instance_descriptor))
        y_pop.append(y_hijo)
        fit_list.append(fit)
        
    for i in range(len(y_pop)):
        y_pop[i]=([y_pop[i], fit_list[i]])
    
    for i in range(tam_pop):    #seleccionar dos individuos
        y_contender1=random.choice(y_pop)
        del y_pop[y_pop.index(y_contender1)]
        y_contender2=random.choice(y_pop)
        del y_pop[y_pop.index(y_contender2)]
        
        if y_contender1[1]>=y_contender2[1]:    #y mantener el que mejor fitness tenga
            new_gen.append(y_contender1[0])
            new_fit_list.append(y_contender1[1])
        else:
            new_gen.append(y_contender2[0])
            new_fit_list.append(y_contender2[1])
    
    return new_gen, new_fit_list

#ALGORITMO GENÉTICO DE SEGUNDO NIVEL, ENCUENTRA Ys, FUNCIONA COMO FUNCIÓN DE FITNESS PARA LAS Xs
def Genetic_y(n, x, sat_instance_descriptor, swarmsize=40, iterations=50, sel=1, Dpc=0, muestreo=1):
    pop=random_x_y(n,swarmsize)
    y_pop=pop[1]
    fit_list=[]
    y_pop_full=[]
    
    for y_padre in y_pop:
        fit=fitness_check_FND(sat_instance_instancer(x, y_padre, sat_instance_descriptor))
        fit_list.append(fit)
    
    if sel:    #selección por elitismo
        if Dpc:   #cruce 2 puntos
            for it in range(iterations):
                y_hijos=cruce_2P(y_pop)
                y_pop, fit_list= selection_elitism_ys(x, y_pop, y_hijos, fit_list, sat_instance_descriptor)
        else:   #cruce 1 punto
            for it in range(iterations):
                y_hijos=cruce_1P(y_pop)
                y_pop, fit_list= selection_elitism_ys(x, y_pop, y_hijos, fit_list, sat_instance_descriptor)
    else:    #selección por torneo
        if Dpc:   #cruce 2 puntos
            for it in range(iterations):
                y_hijos=cruce_2P(y_pop)
                y_pop, fit_list= selection_tourney_ys(x, y_pop, y_hijos, fit_list, sat_instance_descriptor)
        else:    #cruce 1 punto
            for it in range(iterations):
                y_hijos=cruce_1P(y_pop)
                y_pop, fit_list= selection_tourney_ys(x, y_pop, y_hijos, fit_list, sat_instance_descriptor)
    
    for i in range(len(y_pop)):
        y_pop_full.append([y_pop[i], fit_list[i]])
    y_pop_sorted=list(reversed(sorted(y_pop_full, key=lambda ind: ind[1])))
    
    best_ind_y=y_pop_sorted[0][0]
    if(muestreo):
        random_ind_index=random.sample(range(3, swarmsize-1), 3)
        best_fit_y=(y_pop_sorted[0][1]+ y_pop_sorted[1][1] + y_pop_sorted[2][1] + y_pop_sorted[random_ind_index[0]][1] +  y_pop_sorted[random_ind_index[1]][1] + y_pop_sorted[random_ind_index[2]][1]) / 6  
    else:
        best_fit_y=y_pop_sorted[0][1]

    return best_ind_y, best_fit_y

#FUNCIONES DE SELECCIÓN PARA Xs
def selection_elitism_xs(x_pop, x_hijos, fit_list, sat_instance_descriptor, swarmsizeY=40, iterationsY=50, selY=1, DpcY=0):   #función de selección elitista para las xs, cuanto mayor sea el fitness, más cláusulas rompe el individuo y para las variables xs fijas
    new_gen=[]
    new_fit_list=[]
    tam_pop=len(x_pop)
    
    for x_hijo in x_hijos:    #para cada hijo se calcula su fitness
        fit=Genetic_y(n, x_hijo, sat_instance_descriptor, swarmsizeY, iterationsY, selY, DpcY)
        x_pop.append(x_hijo)
        fit_list.append(fit[1])
        
    for i in range(len(x_pop)):    #se ordenan los individuos por su fitness
        x_pop[i]=([x_pop[i], fit_list[i]])
    x_pop_sorted=list(sorted(x_pop, key=lambda ind: ind[1]))
    
    for i in range(tam_pop):   #los individuos x con la mayor fitness sobreviven
       new_gen.append(x_pop_sorted[i][0])
       new_fit_list.append(x_pop_sorted[i][1])
        
    return new_gen, new_fit_list
    
def selection_tourney_xs(x_pop, x_hijos, fit_list, sat_instance_descriptor, swarmsizeY=40, iterationsY=50, selY=1, DpcY=0):   #función de selección por torneo
    new_gen=[]
    new_fit_list=[]
    tam_pop=len(x_pop)
    
    for x_hijo in x_hijos:
        fit=Genetic_y(n, x_hijo, sat_instance_descriptor, swarmsizeY, iterationsY, selY, DpcY)
        x_pop.append(x_hijo)
        fit_list.append(fit[1])

    for i in range(len(x_pop)):
        x_pop[i]=[x_pop[i], fit_list[i]]
    
    for i in range(tam_pop):    #seleccionar dos individuos
        x_contender1=random.choice(x_pop)
        del x_pop[x_pop.index(x_contender1)]
        x_contender2=random.choice(x_pop)
        del x_pop[x_pop.index(x_contender2)]
        
        if x_contender1[1]<=x_contender2[1]:    #y mantener el que mejor fitness tenga
            new_gen.append(x_contender1[0])
            new_fit_list.append(x_contender1[1])
        else:
            new_gen.append(x_contender2[0])
            new_fit_list.append(x_contender2[1])
    
    return new_gen, new_fit_list

#ALGORITMO GENÉTICO DE PRIMER NIVEL, ENCUENTRA Xs
def Genetic_x(n, sat_instance_descriptor, swarmsize=40, iterations=50, swarmsizeY=40, iterationsY=50, sel=1, selY=1, Dpc=0, DpcY=0):
    pop=random_x_y(n,swarmsize)
    x_pop=pop[1]
    fit_list=[]
    x_pop_full=[]
    
    for x_padre in x_pop:
        fit=Genetic_y(n, x_padre, sat_instance_descriptor, swarmsizeY, iterationsY, selY)
        fit_list.append(fit[1])

    if sel:   #selección por elitismo
        if Dpc:   #cruce 2 puntos
            for it in range(iterations):
                x_hijos=cruce_2P(x_pop)
                x_pop, fit_list= selection_elitism_xs(x_pop, x_hijos, fit_list, sat_instance_descriptor, swarmsizeY, iterationsY, selY, DpcY)
        else:    #cruce 1 punto
            for it in range(iterations):
                x_hijos=cruce_1P(x_pop)
                x_pop, fit_list= selection_elitism_xs(x_pop, x_hijos, fit_list, sat_instance_descriptor, swarmsizeY, iterationsY, selY, DpcY)
    else:   #selección por torneo
        if Dpc:   #cruce 2 puntos
            for it in range(iterations):
                x_hijos=cruce_2P(x_pop)
                x_pop, fit_list= selection_tourney_xs(x_pop, x_hijos, fit_list, sat_instance_descriptor, swarmsizeY, iterationsY, selY, DpcY)
        else:    #cruce 1 punto
            for it in range(iterations):
                x_hijos=cruce_1P(x_pop)
                x_pop, fit_list= selection_tourney_xs(x_pop, x_hijos, fit_list, sat_instance_descriptor, swarmsizeY, iterationsY, selY, DpcY)
    
    for i in range(len(x_pop)):
        x_pop_full.append([x_pop[i], fit_list[i]])
    x_pop_sorted=list(sorted(x_pop_full, key=lambda ind: ind[1]))
    
    best_ind_x=x_pop_sorted[0][0]
    best_fit_x=x_pop_sorted[0][1]

    return best_ind_x, best_fit_x

In [23]:
#Funciones para AG con fuerza bruta en el segundo nivel, para control sin varianza

def selection_elitism_xs_no_ag(x_pop, x_hijos, fit_list, sat_instance_descriptor, n=10, m=4):   #función de selección elitista para las xs, el fitness de cada individuo es su validez, que se comprueba por fuerza bruta
    new_gen=[]
    new_fit_list=[]
    tam_pop=len(x_pop)
    
    for x_hijo in x_hijos:    #para cada hijo se calcula su fitness (por fuerza bruta si es válido o no)
        #fit=Genetic_y(n, x_hijo, sat_instance_descriptor, swarmsizeY, iterationsY, selY, DpcY)
        fit=0
        fit_valid=brute_force_QSAT2_simple([x_hijo], sat_instance_descriptor, n, m)
        if(len(fit_valid[2])>0):
            fit=1
        x_pop.append(x_hijo)
        fit_list.append(fit)
        
    for i in range(len(x_pop)):    #se ordenan los individuos por su fitness
        x_pop[i]=([x_pop[i], fit_list[i]])
    x_pop_sorted=list(reversed(sorted(x_pop, key=lambda ind: ind[1])))
    
    for i in range(tam_pop):   #los individuos x con la mayor fitness sobreviven
       new_gen.append(x_pop_sorted[i][0])
       new_fit_list.append(x_pop_sorted[i][1])
        
    return new_gen, new_fit_list
    

#ALGORITMO GENÉTICO DE PRIMER NIVEL, no se ejecuta un algoritmo genético de segundo nivel, sino que el fitness de los inndividuos es su validez comprobada por fuerza bruta
def Genetic_x_2lvl_bruteforce(n, sat_instance_descriptor, swarmsize=40, iterations=50):
    pop=random_x_y(n,swarmsize)
    x_pop=pop[1]
    fit_list=[]
    x_pop_full=[]
    m=len(sat_instance_descriptor[0][-1])
    
    for x_padre in x_pop:
        fit=0
        fit_valid=brute_force_QSAT2_simple([x_padre], sat_instance_descriptor, n, m)
        if(len(fit_valid[2])>0):
            fit=1
        fit_list.append(fit)
   
    for it in range(iterations):
        x_hijos=cruce_1P(x_pop)
        x_pop, fit_list= selection_elitism_xs_no_ag(x_pop, x_hijos, fit_list, sat_instance_descriptor, n)

    for i in range(len(x_pop)):
        x_pop_full.append([x_pop[i], fit_list[i]])
    x_pop_sorted=list(reversed(sorted(x_pop_full, key=lambda ind: ind[1])))
    
    best_ind_x=x_pop_sorted[0][0]
    best_fit_x=x_pop_sorted[0][1]
    #if(x_pop_sorted[0][1]):
    #    best_fit_x=fitness_check_FND(sat_instance_instancer(best_ind_x, best_ind_x, sat_instance_descriptor))
    #else:
    #    best_fit_x=len(sat_instance_descriptor)
    
    return best_ind_x, best_fit_x

In [24]:
#FUNCIONES PARA PRUEBAS DE RATIOS DE VARIABLES Y CLÁUSULAS

def full_tree_exploration_net(xs, ys, n=10, m=3, file=None):   #genera una instancia de QSAT2 y explora todas las combinaciones
    result=[]
    if not file:
        sat_instance_descriptor=sat_instance_generator(n,m)
    else:
        sat_instance_descriptor=load_file_instance(file)
    for x in xs:
        for y in ys:
            sat_instanced=sat_instance_instancer(x, y, sat_instance_descriptor)
            fitness=fitness_check_FND(sat_instanced)
            result.append([x,y,fitness])
    return sat_instance_descriptor, result

def brute_force_QSAT2_FND_net(n=10, m=3, nc=10, file=None):   #genera todos los posibles individuos de x e y, genera un problema QSAT2, lo resuleve para todas las combinaciones y comprueba la satisfactibilidad para las subramas del árbol 
    xs=[]
    x=[]
    a=binstr(n)
    for string in a:
        for i in range(len(string)):
            x.append(int(string[i]))
        xs.append(x)
        x=[]
    ys=xs.copy()
    tree=full_tree_exploration_net(xs, ys, n, m, file)
    resultado=0
    for res in tree[1]:
        if(res[2]<nc):
            resultado+=1
    return tree[0], tree[1], resultado

#Generar 100 instancias de tamaño, ratio y variables por cláusula detetermiado y calcular si son instancias equilibradas
def batch_test_clause_var_ratio_net(n=10, m=3, ratio=1, file_generic_name="QSAT2_FND_instance_n10_c1_net_test", acceptance=0.05, verbose=True):
    nc=int(2*n*ratio)
    len_max=pow(2,n*2)
    file_generic=file_generic_name+".json"
    resultados=[]
    
    for i in range(100):
        sat_instance_descriptor=sat_instance_generator_clause_variabe(n, m, nc)
        save_instance_file(sat_instance_descriptor, file_generic)
        t1=time.time()
        resultado_full_n10_c_net=brute_force_QSAT2_FND_net(n,m,nc,file_generic)
        t2=time.time()
        t=t2-t1
        resultados.append([i, resultado_full_n10_c_net[2], t])
        if(verbose):
            print(resultados[-1])
        if((resultados[-1][1]<(len_max/2 +len_max*acceptance)) and (resultados[-1][1]>(len_max/2 -len_max*acceptance))):
            file_name=file_generic_name+"_"+str(i)+".json"
            save_instance_file(sat_instance_descriptor, file_name)
            file_name2=file_generic_name+"_"+str(i)+"_full_solution.json"
            save_instance_file(resultado_full_n10_c_net[2], file_name2)
            if(verbose):
                print(file_name)
    
    return resultados

#Generar 100 instancias de tamaño y variables por cláusula detetermiado, pero ratio variable entre un intervalo, y calcular si son instancias equilibradas
def batch_test_clause_var_ratio_interval_net(n=10, m=3, ratio=1, ratio_max=2, intervalo=0.02, file_generic="QSAT2_FND_instance_n10_claus_var_net_test_1to2.json", acceptance=0.05, verbose=True):
    resultados=[]   
    while(ratio<=(ratio_max+intervalo)):
        nc=int(2*n*ratio)
        len_max=pow(2,n*2)
        resultados.append([ratio, []])
        sumatorio=0
        contador=0
        for i in range(100):
            sat_instance_descriptor=sat_instance_generator_clause_variabe(n, m, nc)
            save_instance_file(sat_instance_descriptor, file_generic)
            t1=time.time()
            resultado_full_n10_c_net=brute_force_QSAT2_FND_net(n,m,nc,file_generic)
            t2=time.time()
            t=t2-t1
            resultados[-1][1].append([i, resultado_full_n10_c_net[2], t])
            sumatorio+=resultados[-1][1][-1][1]
            if((resultados[-1][1][-1][1]< (len_max/2 + len_max*acceptance)) and (resultados[-1][1][-1][1]> (len_max/2 - len_max*acceptance))):
                contador+=1
    
        resultados[-1].append(contador/100)
        resultados[-1].append(sumatorio/100)
        
        ratio+=intervalo
        
        if(verbose):
            print(resultados[-1])
            
    return resultados


In [25]:
#FUNCIÓN PARA REALIZAR PRUEBAS DE COMBINACIÓN DE HIPERPARÁMETROS (MÉTODOS DE CRUCE Y SELECCIÓN)
def pruebas_combinacion_hiperparametros_FND(n, sat_instance_descriptor, par_comb, n_pruebas=5):
    resultados_totales=[]
    tiempos_medios=[]
    porcentajes_resultados_validos=[]
    sat_size=len(sat_instance_descriptor)
    for hp in par_comb:
        tiempo_total=0
        resultados_validos_totales=0
        resultados_totales.append([])
        for i in range(n_pruebas):
            t1=time.time()
            resultado=Genetic_x(n, sat_instance_descriptor, swarmsize=hp[1][0], iterations=hp[1][1], swarmsizeY=hp[2][0], iterationsY=hp[2][1], sel=hp[0][0], selY=hp[0][1], Dpc=hp[0][2], DpcY=hp[0][3])
            t2=time.time()
            t=t2-t1
            tiempo_total+=t
            resultados_totales[-1].append(resultado)
            if(resultado[1]<sat_size):
                resultados_validos_totales+=1
                
        porcentajes_resultados_validos.append(resultados_validos_totales/n_pruebas)
        tiempos_medios.append(tiempo_total/n_pruebas)

        #print(porcentajes_resultados_validos)
        #print(tiempos_medios)
        #print(resultados_totales)
    return porcentajes_resultados_validos, tiempos_medios, resultados_totales


def pruebas_combinacion_hiperparametros_FND_clause_variable(n, par_comb, n_pruebas=5):
    resultados_totales=[]
    tiempos_medios=[]
    porcentajes_resultados_validos=[]
    for hp in par_comb:
        sat_instance_descriptor=load_file_instance(hp[3])
        sat_size=len(sat_instance_descriptor)
        tiempo_total=0
        resultados_validos_totales=0
        resultados_totales.append([])
        for i in range(n_pruebas):
            t1=time.time()
            resultado=Genetic_x(n, sat_instance_descriptor, swarmsize=hp[1][0], iterations=hp[1][1], swarmsizeY=hp[2][0], iterationsY=hp[2][1], sel=hp[0][0], selY=hp[0][1], Dpc=hp[0][2], DpcY=hp[0][3])
            t2=time.time()
            t=t2-t1
            tiempo_total+=t
            resultados_totales[-1].append(resultado)
            if(resultado[1]<sat_size):
                resultados_validos_totales+=1
                
        porcentajes_resultados_validos.append(resultados_validos_totales/n_pruebas)
        tiempos_medios.append(tiempo_total/n_pruebas)

        #print(porcentajes_resultados_validos)
        #print(tiempos_medios)
        #print(resultados_totales)
    return porcentajes_resultados_validos, tiempos_medios, resultados_totales


#FUNCION PARA EXEPRIMENTOS CON EJECUCIONES DE CONJUNTOS GRANDES DE INTANCIAS DE UN MISMO TAMAÑO DE INSTANCIA (Se calcularon antes las soluciones completas)
def pruebas_qsat2_ag(n, file_names, partial_results_filename, n_pruebas=10, par_comb=[[[1, 1, 0, 0], [40, 50], [40, 50]]], verbose=True):
    resultados_totales=[]
    for h in par_comb:     #para cada par de tamaños de iteraciones del ag inferior
        
        resultados_totales.append([[h[2][0], h[2][1]]])
        if(verbose):
            print("Pop: ", h[2][0], "- Iteraciones: ", h[2][1])
        t1_global=time.time()
        
        for i in range(len(file_names)):   #se ejecutan n_pruebas sobre cada una de las 60 instanicas
            resultados_instancia=[]
            
            file_name=file_names[i]       #carga de la instancia y su solución completa
            file_name_full_soution=file_names[i].replace(".json", "")+"_full_solution.json"
            sat_instance_descriptor=load_file_instance(file_name)
            sat_instance_full_solution=load_file_instance(file_name_full_soution)

            if(verbose):
                print(i, file_name)
                
            resultados_instancia.append(i)
            resultados_instancia.append(file_name)
            tiempo_total=0
            resultados_validos_totales=0

            
            for j in range(n_pruebas):     #prueba unitaria para una instancia (se ejecuta n_pruebas veces)
                valid_solution=0
                
                t1=time.time()
                result_ag=Genetic_x(n, sat_instance_descriptor, swarmsizeY=h[2][0], iterationsY=h[2][1])
                t2=time.time()
                t=t2-t1
                
                tiempo_total+=t
                
                if(result_ag[0] in sat_instance_full_solution):    #comprobación de validez de la solución del ag
                    valid_solution=1
                resultados_validos_totales+=valid_solution
                
                resultados_instancia.append([j, result_ag[0], result_ag[1], valid_solution, t])

                if(verbose):
                    print("- ", resultados_instancia[-1])
    
            porcentaje_resultados_validos=resultados_validos_totales/n_pruebas      #estadísticas del conjunto de pruebas de la instncia
            tiempo_medio=tiempo_total/n_pruebas
            
            resultados_instancia.append(porcentaje_resultados_validos)
            resultados_instancia.append(tiempo_medio)
            resultados_instancia.append(tiempo_total)
    
            t2_global=time.time()
            t_global=t2_global-t1_global
    
            resultados_totales[-1].append(resultados_instancia)
            
            if(verbose):       #resumen de las ejecuciones de la instancia
                print(i+1, "/", len(file_names), " --- ", porcentaje_resultados_validos, " --- ", tiempo_medio, " --- ", t_global, '\n')
    
        save_instance_file(resultados_totales, partial_results_filename)

    return resultados_totales


#FUNCION PARA EXEPRIMENTOS CON EJECUCIONES DE CONJUNTOS GRANDES DE INTANCIAS DE UN MISMO TAMAÑO DE INSTANCIA (no se calcularon antes las soluciones completas)
def pruebas_qsat2_ag_no_full_solution(n, file_names, partial_results_filename, n_pruebas=10, par_comb=[[[1, 1, 0, 0], [40, 50], [40, 50]]], m=4, verbose=True):
    resultados_totales=[]
    for h in par_comb:     #para cada par de tamaños de iteraciones del ag inferior
        
        resultados_totales.append([[h[2][0], h[2][1]]])
        if(verbose):
            print("Pop: ", h[2][0], "- Iteraciones: ", h[2][1])
        t1_global=time.time()
        
        for i in range(len(file_names)):   #se ejecutan n_pruebas sobre cada una de las 60 instanicas
            resultados_instancia=[]
            
            file_name=file_names[i]       #carga de la instancia 
            sat_instance_descriptor=load_file_instance(file_name)

            if(verbose):
                print(i, file_name)
                
            resultados_instancia.append(i)
            resultados_instancia.append(file_name)
            tiempo_total=0
            resultados_validos_totales=0

            valid_solutions_found=[]
            invalid_solutions_found=[]

            
            for j in range(n_pruebas):     #prueba unitaria para una instancia (se ejecuta n_pruebas veces)
                
                t1=time.time()
                result_ag=Genetic_x(n, sat_instance_descriptor, swarmsizeY=h[2][0], iterationsY=h[2][1])
                t2=time.time()
                t=t2-t1
                
                tiempo_total+=t
                
                best_ind_result=result_ag[0]    #comprobación de validez de la solución del ag sin soluciones completas
                if(best_ind_result in valid_solutions_found):
                    valid_solution=1
                elif(best_ind_result in invalid_solutions_found):
                    valid_solution=0
                else:
                    valid_solution=brute_force_QSAT2_simple([best_ind_result], sat_instance_descriptor, n, m)[2]
                    if(valid_solution):
                        valid_solutions_found.append(best_ind_result)
                        valid_solution=1
                    else:
                        invalid_solutions_found.append(best_ind_result)
                        valid_solution=0
                resultados_validos_totales+=valid_solution
                
                resultados_instancia.append([j, result_ag[0], result_ag[1], valid_solution, t])

                if(verbose):
                    print("- ", resultados_instancia[-1])
    
            porcentaje_resultados_validos=resultados_validos_totales/n_pruebas      #estadísticas del conjunto de pruebas de la instncia
            tiempo_medio=tiempo_total/n_pruebas
            
            resultados_instancia.append(porcentaje_resultados_validos)
            resultados_instancia.append(tiempo_medio)
            resultados_instancia.append(tiempo_total)
    
            t2_global=time.time()
            t_global=t2_global-t1_global
    
            resultados_totales[-1].append(resultados_instancia)
            
            if(verbose):       #resumen de las ejecuciones de la instancia
                print(i+1, "/", len(file_names), " --- ", porcentaje_resultados_validos, " --- ", tiempo_medio, " --- ", t_global, '\n')
    
        save_instance_file(resultados_totales, partial_results_filename)

    return resultados_totales


#FUNCION PARA EXEPRIMENTOS CON EJECUCIONES DE AG CON SEGUNDO NIVEL POR FURZA BRUTA (Versión del AG sin varianza, a modo de control)
def pruebas_qsat2_ag_2lvl_bruteforce(n, file_names, n_pruebas=10, par_comb=[[[1, 1, 0, 0], [40, 50], [0, 0]]], verbose=True):
    h=par_comb[0]
    resultados_totales=[]        
    resultados_totales.append([[h[2][0], h[2][1]]])
    if(verbose):
        print("Pop: ", h[2][0], "- Iteraciones: ", h[2][1], " (Segundo nivel por fuerza bruta)")
    t1_global=time.time()
    
    for i in range(len(file_names)):   #se ejecutan n_pruebas sobre cada una de las 60 instanicas
        resultados_instancia=[]
        
        file_name=file_names[i]       #carga de la instancia y su solución completa
        sat_instance_descriptor=load_file_instance(file_name)

        if(verbose):
            print(i, file_name)
            
        resultados_instancia.append(i)
        resultados_instancia.append(file_name)
        tiempo_total=0
        resultados_validos_totales=0

        
        for j in range(n_pruebas):     #prueba unitaria para una instancia (se ejecuta n_pruebas veces)
            
            t1=time.time()
            result_ag=Genetic_x_2lvl_bruteforce(n, sat_instance_descriptor)
            t2=time.time()
            t=t2-t1
            
            tiempo_total+=t
            
            valid_solution=result_ag[1]  #comprobación de validez de la solución del ag (lo hace el propio ag porque el segundo nivel se hace con fuerza bruta)
            resultados_validos_totales+=valid_solution
            
            resultados_instancia.append([j, result_ag[0], valid_solution, t])

            if(verbose):
                print("- ", resultados_instancia[-1])

        porcentaje_resultados_validos=resultados_validos_totales/n_pruebas      #estadísticas del conjunto de pruebas de la instncia
        tiempo_medio=tiempo_total/n_pruebas
        
        resultados_instancia.append(porcentaje_resultados_validos)
        resultados_instancia.append(tiempo_medio)
        resultados_instancia.append(tiempo_total)

        t2_global=time.time()
        t_global=t2_global-t1_global

        resultados_totales[-1].append(resultados_instancia)
        
        if(verbose):       #resumen de las ejecuciones de la instancia
            print(i+1, "/", len(file_names), " --- ", porcentaje_resultados_validos, " --- ", tiempo_medio, " --- ", t_global, '\n')
    
        #save_instance_file(resultados_totales, partial_results_filename)

    return resultados_totales